In [0]:
%pip install boto3
import boto3

In [0]:
%sql
--Reset the Training Environment
DROP SCHEMA IF EXISTS  geography.staging CASCADE;
DROP CATALOG IF EXISTS geography CASCADE;

In [0]:
dbutils.fs.rm("r2://mvcostanzo-geography@4f3914487748b814b84bb931230b53e8.r2.cloudflarestorage.com/unmanaged/", recurse=True)

In [0]:
cloudflareAccessKey = dbutils.secrets.get(scope='cloudflarer2', key='access-key-id')
cloudflareSecretKey = dbutils.secrets.get(scope='cloudflarer2', key='secret-access-key')
r2_client = boto3.client(
    's3',
    endpoint_url= 'https://4f3914487748b814b84bb931230b53e8.r2.cloudflarestorage.com',
    aws_access_key_id = cloudflareAccessKey,
    aws_secret_access_key = cloudflareSecretKey
)

bucket_name = 'mvcostanzo-geography'
directory_prefix = 'catalog/'

objects_to_delete = []
paginator = r2_client.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=bucket_name, Prefix=directory_prefix)

for page in pages:
    if 'Contents' in page:
        for obj in page['Contents']:
            objects_to_delete.append({'Key': obj['Key']})

if objects_to_delete:
    response = r2_client.delete_objects(
        Bucket=bucket_name,
        Delete={'Objects': objects_to_delete}
    )
    print(f"Errors during deletion: {response.get('Errors', [])}")
else:
    print(f"No objects found in '{directory_prefix}' to delete.")

In [0]:
%sql

CREATE CATALOG geography MANAGED LOCATION 'r2://mvcostanzo-geography@4f3914487748b814b84bb931230b53e8.r2.cloudflarestorage.com/catalog';
CREATE SCHEMA geography.staging;
CREATE EXTERNAL VOLUME geography.staging.raw_data
  LOCATION 'r2://mvcostanzo-geography@4f3914487748b814b84bb931230b53e8.r2.cloudflarestorage.com/raw-data';
